In [2]:
import meep as mp
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

# BƯỚC 1 & 2: ĐỌC BẢN QUÉT POLYCAM (FLOOR PLAN) VÀ ÁNH XẠ VẬT LIỆU

In [ ]:
# Kích thước thực tế của trường quét bằng LiDAR
SX, SY = 30.0, 20.0
cell = mp.Vector3(30,20,0)
def create_school_map_from_polycam(image_path=None):
    """
    Nếu đã có ảnh Floorplan xuất từ Polycam, đọc ảnh và chuyển thành ma trận chiết suất.
    Nếu chưa có file ảnh, hàm tự tạo một mặt bằng trường học mẫu (3 phòng + hành lang).
    """
    grid_x, grid_y = 300, 200 # Độ phân giải bản đồ vật liệu
    eps_map = np.ones((grid_x, grid_y)) # Mặc định toàn bộ là không khí (epsilon = 1.0)
    
    if image_path:
        # Đọc ảnh mặt bằng từ Polycam (Chuyển về ảnh xám Grayscale)
        img = Image.open(image_path).convert('L').resize((grid_x, grid_y))
        img_arr = np.array(img).T
        # Ánh xạ: Pixel tối (< 100) là Tường bê tông (eps = 6.25), Pixel vừa là Cửa gỗ/kính (eps = 2.5)
        eps_map[img_arr < 180] = 2.5   # Cửa kính / Vách gỗ
        eps_map[img_arr < 80]  = 6.25  # Tường bê tông cốt thép (n = 2.5 -> eps = n^2 = 6.25)
    else:
        # Mặt bằng giả lập 3 phòng học + 1 hành lang dọc (khi bạn chưa kịp chép ảnh từ iPhone vào WSL)
        eps_map[:, 60:65] = 6.25       # Tường dọc ngăn lớp học và hành lang (dày 0.5m)
        eps_map[100:105, 65:] = 6.25   # Tường ngăn Phòng 1 và Phòng 2
        eps_map[200:205, 65:] = 6.25   # Tường ngăn Phòng 2 và Phòng 3
        # Khoét cửa ra vào (Gỗ/Kính suy hao thấp hơn bê tông)
        eps_map[45:55, 60:65] = 2.25   # Cửa Phòng 1
        eps_map[145:155, 60:65] = 2.25 # Cửa Phòng 2
        eps_map[245:255, 60:65] = 2.25 # Cửa Phòng 3
        
    return eps_map

# Xác định ma trận trích suất từ ảnh của Polycam 
img_path = "data/raw/polycam_floorplan.png"
eps_matrix = create_school_map_from_polycam(image_path=None) # khi nào có ảnh thật thì thay bằng img_path

def polycam_material_func(p):
    # Chuyển tọa độ mét của MEEP (-SX/2 -> SX/2) sang index của ma trận
    ix = int(np.clip((p.x + SX / 2) / SX * eps_matrix.shape[0], 0, eps_matrix.shape[0] - 1))
    iy = int(np.clip((p.y + SY / 2) / SY * eps_matrix.shape[1], 0, eps_matrix.shape[1] - 1))
    return mp.Medium(epsilon=eps_matrix[ix, iy])

# BƯỚC 3: ĐẶT TOPOLOGY LẮP ĐẶT THIẾT BỊ VÔ TUYẾN (TRƯỚC & SAU TỐI ƯU)

In [9]:
def simulate_topology(ap_positions, title):
    sources = [
        mp.Source(
            mp.ContinuousSource(frequency=2.4),
            component=mp.Ez,
            center=mp.Vector3(x, y, 0),
            amplitude=amp # Đại diện cho công suất phát Tx Power (Tier-1/2/3)
        ) for (x, y, amp) in ap_positions
    ]
    
    sim = mp.Simulation(
        cell_size=cell,
        boundary_layers=[mp.PML(1.0)],
        default_material=polycam_material_func,
        sources=sources,
        resolution=6 # Giữ resolution vừa phải (6-10) để chạy nhanh trên laptop
    )
    sim.run(until=120)
    
    # Trích xuất dữ liệu
    dielectric = sim.get_array(center=mp.Vector3(), size=cell, component=mp.Dielectric)
    ez_energy = np.abs(sim.get_array(center=mp.Vector3(), size=cell, component=mp.Ez))**2
    
    # Quy đổi tương đối sang thang đo dB để nhìn rõ Vùng Chết (Dead Zones)
    rssi_db = 10 * np.log10(ez_energy + 1e-6)
    return dielectric, rssi_db

legacy_aps = [(-8, -6, 1.5), (8, -6, 1.5)] 

# Kịch bản M (Đề xuất KHKT 2026): Đưa AP Tier-2 vào trực tiếp tâm lớp học (LOS), đặt 1 AP Tier-3 công suất nhỏ ở hành lang
optimized_aps = [(-10, 3, 1.0), (0, 3, 1.0), (10, 3, 1.0), (0, -6, 0.4)]

# BƯỚC 4 & 5: VẼ BIỂU ĐỒ NHIỆT SO SÁNH & TÁI TINH CHỈNH VỊ TRÍ

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6), dpi=130)

for ax, aps, label in zip(axes, [legacy_aps, optimized_aps], 
                          ["Hiện trạng (N): Lắp đặt ngoài hành lang (Suy hao NLOS & Nhiễu)", 
                           "Tối ưu Topology (M): Quy hoạch LiDAR + Phân tầng Tier-2/Tier-3"]):
    diel, rssi = simulate_topology(aps, label)
    
    # Vẽ khung tường Polycam
    ax.imshow(diel.T, cmap='binary', origin='lower', extent=[-SX/2, SX/2, -SY/2, SY/2])
    # Phủ Heatmap cường độ sóng
    im = ax.imshow(rssi.T, cmap='turbo', alpha=0.75, vmin=-35, vmax=0, origin='lower', extent=[-SX/2, SX/2, -SY/2, SY/2])
    
    # Đánh dấu vị trí lắp đặt Router
    for (x, y, _) in aps:
        ax.scatter(x, y, c='white', edgecolors='black', s=120, marker='^', label='Vị trí AP')
        
    ax.set_title(label, fontsize=11, fontweight='bold')
    ax.set_xlabel("Chiều dài (m)")
    ax.set_ylabel("Chiều rộng (m)")

fig.colorbar(im, ax=axes.ravel().tolist(), label="Cường độ tín hiệu tương đối (dB)")
plt.suptitle("BÀI TOÁN 1: TỐI ƯU HÓA TOPOLOGY VÔ TUYẾN DỰA TRÊN BẢN QUÉT LIDAR POLYCAM & MEEP FDTD", fontsize=13, fontweight='bold')
plt.show()